In [1]:
# ============================================================
# CONFIGURAÇÃO INICIAL DA EXPLORAÇÃO
# ============================================================
#
# Este notebook realiza a exploração local dos arquivos baixados
# e extraídos no notebook 00_aquisicao_dados.ipynb.
#
# Nesta etapa, não são carregadas credenciais da AWS. A exploração
# utiliza apenas arquivos locais presentes em data_lake/external/extraidos.
#
# Entrada:
#
#   .env                         Arquivo usado para localizar a raiz do projeto.
#   data_lake/external/extraidos Diretório com os arquivos extraídos.
#
# Saída:
#
#   PROJECT_ROOT    Caminho da raiz do projeto.
#   EXTRAIDOS_DIR   Caminho dos arquivos disponíveis para exploração.
#   PARAMS_LEITURA  Parâmetros padrão para leitura dos CSVs do INEP.
#
# ============================================================
# BIBLIOTECAS
# ============================================================

import pandas as pd
from pathlib import Path
from dotenv import find_dotenv

# ============================================================
# RAIZ DO PROJETO E DIRETÓRIO DOS DADOS
# ============================================================

# Localiza o arquivo .env a partir do diretório atual do notebook.
# Isso evita caminhos fixos e permite que cada integrante execute
# o projeto em sua própria máquina.
DOTENV = find_dotenv(usecwd=True)

# Interrompe a execução caso o notebook não esteja sendo executado
# dentro da estrutura esperada do projeto.
assert DOTENV, ".env não encontrado. Abra o notebook a partir da raiz do projeto."

# Define a raiz do projeto a partir da localização do arquivo .env.
PROJECT_ROOT = Path(DOTENV).parent

# Diretório onde estão os arquivos extraídos na etapa de aquisição.
EXTRAIDOS_DIR = PROJECT_ROOT / "data_lake" / "external" / "extraidos"

# ============================================================
# PARÂMETROS PADRÃO DE LEITURA
# ============================================================

# Os CSVs do INEP usam separador ';' e encoding ISO-8859-1.
# Esses parâmetros serão reutilizados nas leituras feitas ao longo
# deste notebook.
PARAMS_LEITURA = {
    "sep": ";",
    "encoding": "ISO-8859-1",
    "low_memory": False,
}

# ============================================================
# CONFERÊNCIA DA CONFIGURAÇÃO
# ============================================================

print("Projeto:", PROJECT_ROOT)

Projeto: /mnt/d/diego/01_projects/postech-challenge-2


## 2023

Nesta seção, exploramos os arquivos extraídos referentes ao ano de 2023.

O objetivo é verificar estrutura, quantidade de linhas, colunas disponíveis, tipos de dados e uma pequena amostra dos registros antes das próximas etapas da pipeline.

In [43]:
# ============================================================
# EXPLORAÇÃO DO ARQUIVO DE MUNICÍPIOS - 2023
# ============================================================
#
# Esta célula carrega o arquivo TS_MUNICIPIO.csv de 2023 e exibe
# uma visão inicial da sua estrutura.
#
# A análise inclui:
#
#   - quantidade de linhas;
#   - quantidade de colunas;
#   - tipos de dados;
#   - contagem de valores não nulos;
#   - uso de memória;
#   - primeiras linhas do arquivo.
#
# Entrada:
#
#   data_lake/external/extraidos/2023/TS_MUNICIPIO.csv
#
# Saída:
#
#   df_mun  DataFrame com os dados de municípios de 2023.
#
# ============================================================
# LEITURA DO ARQUIVO
# ============================================================

# Monta o caminho do arquivo de municípios de 2023.
caminho_municipio = EXTRAIDOS_DIR / "2023" / "TS_MUNICIPIO.csv"

# Lê o CSV usando os parâmetros padrão definidos no início do notebook.
df_mun = pd.read_csv(caminho_municipio, **PARAMS_LEITURA)

# ============================================================
# INSPEÇÃO INICIAL
# ============================================================

# Exibe o volume básico do arquivo.
print(f"Linhas : {len(df_mun):,}")
print(f"Colunas: {df_mun.shape[1]}")

# Exibe schema, quantidade de não nulos por coluna e uso de memória.
print("\n--- info(): schema, não nulos e memória ---")
df_mun.info()

# Exibe uma amostra inicial dos registros.
print("\n--- Primeiras 5 linhas ---")
df_mun.head()

Linhas : 11,547
Colunas: 9

--- info(): schema, não nulos e memória ---
<class 'pandas.DataFrame'>
RangeIndex: 11547 entries, 0 to 11546
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   NU_ANO_AVALIACAO       11547 non-null  int64  
 1   CO_UF                  11547 non-null  int64  
 2   SG_UF                  11547 non-null  str    
 3   CO_MUNICIPIO           11547 non-null  int64  
 4   NO_MUNICIPIO           11547 non-null  str    
 5   TP_SERIE               11547 non-null  int64  
 6   ID_TIPO_REDE           11547 non-null  int64  
 7   PC_ALUNO_ALFABETIZADO  11547 non-null  float64
 8   VL_MEDIA_LP            11547 non-null  float64
dtypes: float64(2), int64(5), str(2)
memory usage: 971.1 KB

--- Primeiras 5 linhas ---


,NU_ANO_AVALIACAO,CO_UF,SG_UF,CO_MUNICIPIO,NO_MUNICIPIO,TP_SERIE,ID_TIPO_REDE,PC_ALUNO_ALFABETIZADO,VL_MEDIA_LP
0,2023,11,RO,1100015,Alta Floresta D'Oeste,2,5,64.55,758.3304
1,2023,11,RO,1100015,Alta Floresta D'Oeste,2,3,64.55,758.3304
2,2023,11,RO,1100023,Ariquemes,2,3,62.30,757.0999
3,2023,11,RO,1100023,Ariquemes,2,5,62.30,757.0999
4,2023,11,RO,1100031,Cabixi,2,5,69.10,767.8763


In [44]:
# ============================================================
# EXPLORAÇÃO DO ARQUIVO DE ALUNOS - 2023
# ============================================================
#
# Esta célula carrega o arquivo TS_ALUNO.csv de 2023 e exibe uma
# visão inicial da sua estrutura.
#
# Diferente do arquivo de municípios, este arquivo possui volume
# maior de registros. A leitura abaixo é completa, ou seja, todos
# os dados são carregados em memória pelo pandas.
#
# Observação:
#
#   Para uma inspeção rápida, é possível limitar a leitura com:
#
#   pd.read_csv(caminho_aluno, nrows=5000, **PARAMS_LEITURA)
#
# A análise dos valores não nulos é importante nesta base, pois
# ajuda a identificar colunas de proficiência vazias em registros
# de alunos ausentes.
#
# Entrada:
#
#   data_lake/external/extraidos/2023/TS_ALUNO.csv
#
# Saída:
#
#   df_aluno  DataFrame com os dados de alunos de 2023.
#
# ============================================================
# LEITURA DO ARQUIVO
# ============================================================

# Monta o caminho do arquivo de alunos de 2023.
caminho_aluno = EXTRAIDOS_DIR / "2023" / "TS_ALUNO.csv"

# Lê o CSV completo usando os parâmetros padrão definidos no início do notebook.
# A leitura completa deve ficar para etapas de processamento em lote.
df_aluno = pd.read_csv(caminho_aluno, nrows=5000, **PARAMS_LEITURA)

# ============================================================
# INSPEÇÃO INICIAL
# ============================================================

# Exibe o volume básico do arquivo.
print(f"Linhas : {len(df_aluno):,}")
print(f"Colunas: {df_aluno.shape[1]}")

# Exibe schema, quantidade de não nulos por coluna e uso de memória.
# A diferença de não nulos entre colunas ajuda a identificar campos
# vazios associados a alunos ausentes, como proficiências não calculadas.
print("\n--- info(): repare a diferença de não nulos entre colunas ---")
df_aluno.info()

# Exibe uma amostra inicial dos registros.
print("\n--- Primeiras 5 linhas ---")
df_aluno.head()

Linhas : 5,000
Colunas: 15

--- info(): repare a diferença de não nulos entre colunas ---
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   NU_ANO_AVALIACAO     5000 non-null   int64  
 1   CO_UF                5000 non-null   int64  
 2   SG_UF                5000 non-null   str    
 3   ID_ALUNO             5000 non-null   int64  
 4   TP_SERIE             5000 non-null   int64  
 5   ID_ESCOLA            5000 non-null   int64  
 6   TP_DEPENDENCIA       5000 non-null   int64  
 7   CO_MUNICIPIO         5000 non-null   int64  
 8   NO_MUNICIPIO         5000 non-null   str    
 9   IN_PRESENCA_LP       5000 non-null   int64  
 10  IN_PREENCHIMENTO_LP  5000 non-null   int64  
 11  CO_CADERNO_LP        5000 non-null   int64  
 12  VL_PESO_ALUNO_LP     4389 non-null   float64
 13  VL_PROFICIENCIA_LP   4389 non-null   float64
 14  IN_ALFABE

,NU_ANO_AVALIACAO,CO_UF,SG_UF,ID_ALUNO,TP_SERIE,ID_ESCOLA,TP_DEPENDENCIA,CO_MUNICIPIO,NO_MUNICIPIO,IN_PRESENCA_LP,IN_PREENCHIMENTO_LP,CO_CADERNO_LP,VL_PESO_ALUNO_LP,VL_PROFICIENCIA_LP,IN_ALFABETIZADO
0,2023,11,RO,11008701,2,60000001,3,1100205,Porto Velho,0,0,17,NaN,NaN,0
1,2023,11,RO,11008695,2,60000001,3,1100205,Porto Velho,1,1,17,1.045465,714.314857,0
2,2023,11,RO,11008687,2,60000001,3,1100205,Porto Velho,0,0,17,NaN,NaN,0
3,2023,11,RO,11008682,2,60000001,3,1100205,Porto Velho,1,1,17,1.045465,759.206313,1
4,2023,11,RO,11008729,2,60000001,3,1100205,Porto Velho,0,0,19,NaN,NaN,0


In [4]:
# ============================================================
# EXPLORAÇÃO DO ARQUIVO DE ESTADOS - 2023
# ============================================================
#
# Esta célula carrega o arquivo TS_ESTADO.csv de 2023 e exibe uma
# visão inicial dos dados agregados por unidade federativa.
#
# Além da quantidade de linhas e colunas, calculamos um resumo
# estatístico apenas para variáveis numéricas que representam medidas
# reais de desempenho.
#
# Observação:
#
#   Algumas colunas numéricas são códigos, como CO_UF, TP_SERIE e
#   ID_TIPO_REDE. Embora o pandas consiga calcular média e desvio
#   dessas colunas, esses resultados não têm significado analítico.
#
# Entrada:
#
#   data_lake/external/extraidos/2023/TS_ESTADO.csv
#
# Saída:
#
#   df_est  DataFrame com os dados de estados de 2023.
#
# ============================================================
# LEITURA DO ARQUIVO
# ============================================================

# Monta o caminho do arquivo de estados de 2023.
caminho_estado = EXTRAIDOS_DIR / "2023" / "TS_ESTADO.csv"

# Lê o CSV usando os parâmetros padrão definidos no início do notebook.
df_est = pd.read_csv(caminho_estado, **PARAMS_LEITURA)

# ============================================================
# INSPEÇÃO INICIAL
# ============================================================

# Exibe o volume básico do arquivo.
print(f"Linhas : {len(df_est):,}")
print(f"Colunas: {df_est.shape[1]}")

# ============================================================
# RESUMO ESTATÍSTICO
# ============================================================

# Seleciona apenas colunas que representam medidas analíticas reais.
# Códigos numéricos são excluídos porque média, mínimo e máximo desses
# campos não representam uma informação estatística útil.
COLS_MEDIDA = ["PC_ALUNO_ALFABETIZADO", "VL_MEDIA_LP"]

print("\n--- Resumo estatístico: apenas medidas ---")
display(df_est[COLS_MEDIDA].describe())

# Exibe uma amostra inicial dos registros.
print("\n--- Primeiras 5 linhas ---")
display(df_est.head())

Linhas : 70
Colunas: 7

--- Resumo estatístico: apenas medidas ---


,PC_ALUNO_ALFABETIZADO,VL_MEDIA_LP
count,70.000000,70.000000
mean,55.319000,744.314474
std,13.059867,16.147667
min,30.570000,712.562000
25%,44.960000,733.381700
50%,54.415000,744.668700
75%,63.520000,751.650700
max,84.490000,795.729300



--- Primeiras 5 linhas ---


,NU_ANO_AVALIACAO,CO_UF,SG_UF,TP_SERIE,ID_TIPO_REDE,PC_ALUNO_ALFABETIZADO,VL_MEDIA_LP
0,2023,11,RO,2,2,58.65,751.4731
1,2023,11,RO,2,3,65.17,760.1971
2,2023,11,RO,2,5,64.60,759.4357
3,2023,13,AM,2,3,49.20,733.6637
4,2023,13,AM,2,5,52.20,736.4687


In [45]:
# ============================================================
# EXPLORAÇÃO DA PLANILHA DE METAS MUNICIPAIS - 2023
# ============================================================
#
# Esta célula carrega a planilha XLSX de resultados e metas dos
# municípios em 2023.
#
# A planilha possui duas linhas de cabeçalho:
#
#   - primeira linha: rótulos descritivos;
#   - segunda linha: nomes técnicos das colunas, seguindo padrão
#     semelhante aos CSVs do INEP.
#
# Por isso, usamos header=1 para considerar a segunda linha como
# cabeçalho do DataFrame.
#
# Também removemos linhas finais de rodapé, como observações e linhas
# em branco, mantendo apenas registros com CO_UF preenchido.
#
# Entrada:
#
#   data_lake/external/extraidos/2023/resultados_e_metas_municipios.xlsx
#
# Saída:
#
#   df_mun_meta  DataFrame com resultados e metas municipais de 2023.
#
# ============================================================
# LEITURA DA PLANILHA
# ============================================================

# Monta o caminho da planilha de metas municipais de 2023.
caminho_xlsx = EXTRAIDOS_DIR / "2023" / "resultados_e_metas_municipios.xlsx"

# Lê a aba de divulgação municipal.
# O header=1 usa a segunda linha da planilha como nome das colunas.
# O dtype=str preserva códigos com zeros à esquerda, quando existirem.
df_mun_meta = pd.read_excel(
    caminho_xlsx,
    sheet_name="Divulgação Alfabet Municipio",
    header=1,
    dtype=str
)

# ============================================================
# AJUSTE DE RODAPÉ
# ============================================================

# Remove linhas finais que não representam municípios, como linhas
# em branco ou observações no rodapé da planilha.
df_mun_meta = df_mun_meta[df_mun_meta["CO_UF"].notna()].copy()

# ============================================================
# INSPEÇÃO INICIAL
# ============================================================

# Exibe o volume básico da planilha após o ajuste de rodapé.
print(f"Linhas : {len(df_mun_meta):,}")
print(f"Colunas: {df_mun_meta.shape[1]}")

# Exibe os nomes das colunas identificadas na planilha.
print("\nColunas:", list(df_mun_meta.columns))

# Exibe uma amostra inicial dos registros.
df_mun_meta.head()

Linhas : 5,468
Colunas: 16

Colunas: ['ANO', 'CO_UF', 'SG_UF', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'NO_TP_REDE', 'PC_ALUNO_ALFABETIZADO', 'META_FINAL_2024', 'META_FINAL_2025', 'META_FINAL_2026', 'META_FINAL_2027', 'META_FINAL_2028', 'META_FINAL_2029', 'META_FINAL_2030', 'NIVEIS_ALFABETIZACAO_2023', 'PC_AVALIADOS_LP']


,ANO,CO_UF,SG_UF,CO_MUNICIPIO,NO_MUNICIPIO,NO_TP_REDE,PC_ALUNO_ALFABETIZADO,META_FINAL_2024,META_FINAL_2025,META_FINAL_2026,META_FINAL_2027,META_FINAL_2028,META_FINAL_2029,META_FINAL_2030,NIVEIS_ALFABETIZACAO_2023,PC_AVALIADOS_LP
0,2023,11,RO,1100015,Alta Floresta D'Oeste,MUNICIPAL,64.55,67.07860126116485,69.5120284348663,71.84109325785958,74.05863395650297,76.159493418938,78.14043382279495,80,3,89.37
1,2023,11,RO,1100023,Ariquemes,MUNICIPAL,62.3,65.21687842070342,68.02390876920695,70.7061609144921,73.25188946172531,75.65256653626597,77.90277676917654,80,3,89.79
2,2023,11,RO,1100031,Cabixi,MUNICIPAL,69.1,70.84502912613192,72.53068615333729,74.15443845377203,75.71434639019579,77.20904206573897,78.63769980684307,80,3,90.48
3,2023,11,RO,1100049,Cacoal,MUNICIPAL,62.51,65.39071596861464,68.16281698785075,70.81199011575661,73.32698568453318,75.69964198045608,77.92478110638412,80,3,84.44
4,2023,11,RO,1100056,Cerejeiras,MUNICIPAL,58.53,62.09040103983767,65.52517196826541,68.80509082659395,71.9067479925146,74.81290415754658,77.51244530506875,80,2,92.12


In [46]:
# ============================================================
# EXPLORAÇÃO DO DICIONÁRIO DE VARIÁVEIS - 2023
# ============================================================
#
# Esta célula carrega a aba de dicionário de variáveis da planilha
# de resultados e metas municipais de 2023.
#
# O objetivo é consultar o significado dos campos presentes na
# planilha, facilitando a interpretação das colunas durante a
# exploração e nas próximas etapas da pipeline.
#
# Entrada:
#
#   caminho_xlsx  Caminho da planilha de metas municipais de 2023.
#   Aba:          variáveis
#
# Saída:
#
#   dic  DataFrame com nome do campo e descrição da variável.
#
# ============================================================
# LEITURA DO DICIONÁRIO
# ============================================================

# Lê apenas as colunas que contêm o nome técnico do campo e sua descrição.
# Como essa aba não possui cabeçalho estruturado para leitura direta,
# usamos header=None e definimos os nomes das colunas manualmente.
dic = pd.read_excel(
    caminho_xlsx,
    sheet_name="variáveis",
    header=None,
    usecols=[1, 2],
    names=["campo", "descricao"]
).dropna(how="all")

# ============================================================
# VISUALIZAÇÃO
# ============================================================

# Exibe o dicionário de variáveis para apoiar a interpretação da planilha.
dic

,campo,descricao
1,ANO DA AVALIAÇÃO,Ano de realização da avaliação
2,CÓDIGO UF,Código da UF a que pertence o município
3,SIGLA UF,Sigla da UF a que pertence o município
4,CÓDIGO MUNICÍPIO,Código do município
5,NOME DO MUNICÍPIO,Nome do município
6,REDE,Rede de realização: todos os resultados se ref...
7,PERCENTUAL DE ALUNOS ALFABETIZADOS (1),Percentual de alunos alfabetizados: percentual...
8,META 2024 (2),Meta estabelecida para o município em 2024
9,META 2025,Meta estabelecida para o município em 2025
10,META 2026,Meta estabelecida para o município em 2026


In [48]:
# ============================================================
# EXPLORAÇÃO DA PLANILHA DE METAS POR UF - 2023
# ============================================================
#
# Esta célula carrega a planilha XLSX de resultados e metas por
# unidade federativa em 2023.
#
# A planilha possui duas linhas de cabeçalho:
#
#   - primeira linha: rótulos descritivos;
#   - segunda linha: nomes técnicos das colunas.
#
# Por isso, usamos header=1 para considerar a segunda linha como
# cabeçalho do DataFrame.
#
# Também removemos linhas finais de rodapé, como observações e linhas
# em branco, mantendo apenas registros com CD_UF preenchido.
#
# Entrada:
#
#   data_lake/external/extraidos/2023/resultados_e_metas_ufs.xlsx
#
# Saída:
#
#   df_uf_meta  DataFrame com resultados e metas por UF de 2023.
#
# ============================================================
# LEITURA DA PLANILHA
# ============================================================

# Monta o caminho da planilha de metas por UF de 2023.
caminho_xlsx = EXTRAIDOS_DIR / "2023" / "resultados_e_metas_ufs.xlsx"

# Lê a aba de divulgação por UF e Brasil.
# O header=1 usa a segunda linha da planilha como nome das colunas.
# O dtype=str preserva códigos com zeros à esquerda, quando existirem.
df_uf_meta = pd.read_excel(
    caminho_xlsx,
    sheet_name="Divulgação Alfabet UF e Brasil",
    header=1,
    dtype=str
)

# ============================================================
# AJUSTE DE RODAPÉ
# ============================================================

# Remove linhas finais que não representam registros analíticos,
# como linhas em branco ou observações no rodapé da planilha.
df_uf_meta = df_uf_meta[df_uf_meta["CD_UF"].notna()].copy()

# ============================================================
# INSPEÇÃO INICIAL
# ============================================================

# Exibe o volume básico da planilha após o ajuste de rodapé.
print(f"Linhas : {len(df_uf_meta):,}")
print(f"Colunas: {df_uf_meta.shape[1]}")

# Exibe os nomes das colunas identificadas na planilha.
print("\nColunas:", list(df_uf_meta.columns))

# Exibe uma amostra inicial dos registros.
df_uf_meta.head()

Linhas : 27
Colunas: 16

Colunas: ['ANO', 'CD_UF', 'SIGLA_UF', 'NOME_UF', 'REDE', 'SAEB_2019', 'SAEB_2021', 'PC_ALUNO_ALFABETIZADO', 'META_FINAL_2024', 'META_FINAL_2025', 'META_FINAL_2026', 'META_FINAL_2027', 'META_FINAL_2028', 'META_FINAL_2029', 'META_FINAL_2030', 'PC_AVALIADOS_LP']


,ANO,CD_UF,SIGLA_UF,NOME_UF,REDE,SAEB_2019,SAEB_2021,PC_ALUNO_ALFABETIZADO,META_FINAL_2024,META_FINAL_2025,META_FINAL_2026,META_FINAL_2027,META_FINAL_2028,META_FINAL_2029,META_FINAL_2030,PC_AVALIADOS_LP
1,2023,12,AC,Acre,PÚBLICA,52.87,20.05,-,-,-,-,-,-,-,-,-
2,2023,27,AL,Alagoas,PÚBLICA,39.01,30.04,43.88,49.699999999999996,55.5,61.1,66.5,71.5,76,> 80,92.36
3,2023,13,AM,Amazonas,PÚBLICA,43.83,28.76,52.2,56.8,61.3,65.6,69.6,73.4,76.9,> 80,76.14
4,2023,16,AP,Amapá,PÚBLICA,24.77,18.77,41.56,47.599999999999994,53.8,59.900000000000006,65.6,70.9,75.8,> 80,89.68
5,2023,29,BA,Bahia,PÚBLICA,41.36,24.49,36.8,43.4,50.199999999999996,57.1,63.6,69.80000000000001,75.19999999999999,> 80,84.53


In [49]:
# ============================================================
# EXPLORAÇÃO DO DICIONÁRIO DE VARIÁVEIS POR UF - 2023
# ============================================================
#
# Esta célula carrega a aba de dicionário de variáveis da planilha
# de resultados e metas por unidade federativa em 2023.
#
# O objetivo é consultar o significado dos campos presentes na
# planilha de UFs, apoiando a interpretação das colunas durante
# a exploração dos dados.
#
# Entrada:
#
#   caminho_xlsx  Caminho da planilha de metas por UF de 2023.
#   Aba:          variáveis
#
# Saída:
#
#   dic_uf  DataFrame com nome da variável e descrição.
#
# ============================================================
# LEITURA DO DICIONÁRIO
# ============================================================

# Lê as colunas C e D, onde estão o nome da variável e sua descrição.
# Como essa aba não possui cabeçalho estruturado para leitura direta,
# usamos header=None e renomeamos as colunas após a leitura.
dic_uf = (
    pd.read_excel(
        caminho_xlsx,
        sheet_name="variáveis",
        header=None,
        usecols="C:D"
    )
    .dropna(how="all")
    .rename(columns={2: "variavel", 3: "descricao"})
    .reset_index(drop=True)
)

# ============================================================
# VISUALIZAÇÃO
# ============================================================

# Exibe o dicionário de variáveis para apoiar a interpretação da planilha.
dic_uf

,variavel,descricao
0,ANO DA AVALIAÇÃO,Ano de realização da avaliação
1,CÓDIGO UF,Código da UF
2,SIGLA UF,Sigla da UF
3,NOME UF,Nome da UF
4,REDE,Rede de realização: todos os resultados se ref...
5,PERCENTUAL DE ALUNOS ALFABETIZADOS Saeb 2019,Percentual de alunos alfabetizados Saeb 2019: ...
6,PERCENTUAL DE ALUNOS ALFABETIZADOS Saeb 2021,Percentual de alunos alfabetizados Saeb 2021: ...
7,PERCENTUAL DE ALUNOS ALFABETIZADOS Sistemas es...,Percentual de alunos alfabetizados Sistemas es...
8,META 2024 (2),Meta estabelecida para o estado ou Brasil em 2024
9,META 2025,Meta estabelecida para o estado ou Brasil em 2025


## 2024

Nesta seção, repetimos a exploração inicial para os arquivos referentes ao ano de 2024, usando a mesma lógica aplicada em 2023.

In [50]:
# ============================================================
# EXPLORAÇÃO DO ARQUIVO DE MUNICÍPIOS - 2024
# ============================================================
#
# Carrega o arquivo TS_MUNICIPIO.csv de 2024 e exibe volume,
# estrutura, tipos de dados e uma amostra inicial.
#
# ============================================================
# LEITURA DO ARQUIVO
# ============================================================

caminho_municipio = EXTRAIDOS_DIR / "2024" / "TS_MUNICIPIO.csv"
df_mun = pd.read_csv(caminho_municipio, **PARAMS_LEITURA)

# ============================================================
# INSPEÇÃO INICIAL
# ============================================================

print(f"Linhas : {len(df_mun):,}")
print(f"Colunas: {df_mun.shape[1]}")

print("\n--- info(): schema, não nulos e memória ---")
df_mun.info()

print("\n--- Primeiras 5 linhas ---")
df_mun.head()

Linhas : 12,448
Colunas: 18

--- info(): schema, não nulos e memória ---
<class 'pandas.DataFrame'>
RangeIndex: 12448 entries, 0 to 12447
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   NU_ANO_AVALIACAO       12448 non-null  int64  
 1   CO_UF                  12448 non-null  int64  
 2   SG_UF                  12448 non-null  str    
 3   CO_MUNICIPIO           12448 non-null  int64  
 4   NO_MUNICIPIO           12448 non-null  str    
 5   TP_SERIE               12448 non-null  int64  
 6   ID_TIPO_REDE           12448 non-null  int64  
 7   PC_ALUNO_ALFABETIZADO  12448 non-null  float64
 8   VL_MEDIA_LP            12448 non-null  float64
 9   PC_ALUNO_NIVEL_0_LP    12448 non-null  float64
 10  PC_ALUNO_NIVEL_1_LP    12448 non-null  float64
 11  PC_ALUNO_NIVEL_2_LP    12448 non-null  float64
 12  PC_ALUNO_NIVEL_3_LP    12448 non-null  float64
 13  PC_ALUNO_NIVEL_4_LP    12448 non-null  float

,NU_ANO_AVALIACAO,CO_UF,SG_UF,CO_MUNICIPIO,NO_MUNICIPIO,TP_SERIE,ID_TIPO_REDE,PC_ALUNO_ALFABETIZADO,VL_MEDIA_LP,PC_ALUNO_NIVEL_0_LP,PC_ALUNO_NIVEL_1_LP,PC_ALUNO_NIVEL_2_LP,PC_ALUNO_NIVEL_3_LP,PC_ALUNO_NIVEL_4_LP,PC_ALUNO_NIVEL_5_LP,PC_ALUNO_NIVEL_6_LP,PC_ALUNO_NIVEL_7_LP,PC_ALUNO_NIVEL_8_LP
0,2024,11,RO,1100015,Alta Floresta D'Oeste,2,3,67.79,752.50,1.41,1.49,3.60,5.16,31.95,36.81,14.50,4.02,1.06
1,2024,11,RO,1100015,Alta Floresta D'Oeste,2,5,67.79,752.50,1.41,1.49,3.60,5.16,31.95,36.81,14.50,4.02,1.06
2,2024,11,RO,1100023,Ariquemes,2,5,65.62,748.16,1.99,3.23,6.04,6.83,28.27,33.89,15.64,3.11,1.01
3,2024,11,RO,1100023,Ariquemes,2,3,65.62,748.16,1.99,3.23,6.04,6.83,28.27,33.89,15.64,3.11,1.01
4,2024,11,RO,1100031,Cabixi,2,3,75.88,759.66,0.00,1.31,2.62,3.96,27.16,43.96,13.11,5.23,2.65


In [52]:
# ============================================================
# EXPLORAÇÃO DO ARQUIVO DE ALUNOS - 2024
# ============================================================
#
# Carrega uma amostra do arquivo TS_ALUNO.csv de 2024 para inspeção
# inicial de estrutura, tipos de dados e primeiras linhas.
#
# Como este arquivo possui grande volume de registros, a leitura
# completa pode consumir muita memória, especialmente se outros anos
# já tiverem sido carregados no notebook.
#
# ============================================================
# LEITURA DO ARQUIVO
# ============================================================

caminho_aluno = EXTRAIDOS_DIR / "2024" / "TS_ALUNO.csv"

# Lê apenas uma amostra para exploração inicial.
# A leitura completa deve ficar para etapas de processamento em lote.
df_aluno = pd.read_csv(caminho_aluno, nrows=5000, **PARAMS_LEITURA)

# ============================================================
# INSPEÇÃO INICIAL
# ============================================================

print(f"Linhas lidas : {len(df_aluno):,}")
print(f"Colunas     : {df_aluno.shape[1]}")

print("\n--- info(): estrutura da amostra ---")
df_aluno.info()

print("\n--- Primeiras 5 linhas ---")
df_aluno.head()

Linhas lidas : 5,000
Colunas     : 15

--- info(): estrutura da amostra ---
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   NU_ANO_AVALIACAO     5000 non-null   int64  
 1   CO_UF                5000 non-null   int64  
 2   SG_UF                5000 non-null   str    
 3   ID_ALUNO             5000 non-null   int64  
 4   TP_SERIE             5000 non-null   int64  
 5   ID_ESCOLA            5000 non-null   int64  
 6   TP_DEPENDENCIA       5000 non-null   int64  
 7   CO_MUNICIPIO         5000 non-null   int64  
 8   NO_MUNICIPIO         5000 non-null   str    
 9   IN_PRESENCA_LP       5000 non-null   int64  
 10  IN_PREENCHIMENTO_LP  5000 non-null   int64  
 11  CO_CADERNO_LP        5000 non-null   int64  
 12  VL_PESO_ALUNO_LP     4538 non-null   float64
 13  VL_PROFICIENCIA_LP   4538 non-null   float64
 14  IN_ALFABETIZADO      50

,NU_ANO_AVALIACAO,CO_UF,SG_UF,ID_ALUNO,TP_SERIE,ID_ESCOLA,TP_DEPENDENCIA,CO_MUNICIPIO,NO_MUNICIPIO,IN_PRESENCA_LP,IN_PREENCHIMENTO_LP,CO_CADERNO_LP,VL_PESO_ALUNO_LP,VL_PROFICIENCIA_LP,IN_ALFABETIZADO
0,2024,11,RO,11022231,2,60000163,3,1100015,Alta Floresta D'Oeste,1,1,1,1.06,759.43,1
1,2024,11,RO,11019498,2,60000164,3,1100015,Alta Floresta D'Oeste,1,1,10,1.00,741.28,0
2,2024,11,RO,11019621,2,60000334,3,1100015,Alta Floresta D'Oeste,1,1,9,1.13,807.04,1
3,2024,11,RO,11019638,2,60000334,3,1100015,Alta Floresta D'Oeste,1,1,19,1.09,730.96,0
4,2024,11,RO,11019694,2,60000379,3,1100015,Alta Floresta D'Oeste,1,1,8,1.11,735.46,0


In [53]:
# ============================================================
# EXPLORAÇÃO DO ARQUIVO DE ESTADOS - 2024
# ============================================================
#
# Carrega o arquivo TS_ESTADO.csv de 2024 e exibe volume,
# resumo estatístico das medidas reais e uma amostra inicial.
#
# ============================================================
# LEITURA DO ARQUIVO
# ============================================================

caminho_estado = EXTRAIDOS_DIR / "2024" / "TS_ESTADO.csv"
df_est = pd.read_csv(caminho_estado, **PARAMS_LEITURA)

# ============================================================
# INSPEÇÃO INICIAL
# ============================================================

print(f"Linhas : {len(df_est):,}")
print(f"Colunas: {df_est.shape[1]}")

# ============================================================
# RESUMO ESTATÍSTICO
# ============================================================

# Seleciona apenas medidas analíticas reais.
# Códigos numéricos, como CO_UF, TP_SERIE e ID_TIPO_REDE,
# não possuem média ou desvio com significado analítico.
COLS_MEDIDA = ["PC_ALUNO_ALFABETIZADO", "VL_MEDIA_LP"]

print("\n--- Resumo estatístico: apenas medidas ---")
display(df_est[COLS_MEDIDA].describe())

print("\n--- Primeiras 5 linhas ---")
display(df_est.head())

Linhas : 75
Colunas: 16

--- Resumo estatístico: apenas medidas ---


,PC_ALUNO_ALFABETIZADO,VL_MEDIA_LP
count,75.000000,75.000000
mean,57.358667,746.290661
std,13.322121,15.486466
min,34.460000,723.060000
25%,47.870000,735.238500
50%,56.020000,745.520000
75%,64.655000,754.725000
max,86.210000,797.340000



--- Primeiras 5 linhas ---


,NU_ANO_AVALIACAO,CO_UF,SG_UF,TP_SERIE,ID_TIPO_REDE,PC_ALUNO_ALFABETIZADO,VL_MEDIA_LP,PC_ALUNO_NIVEL_0_LP,PC_ALUNO_NIVEL_1_LP,PC_ALUNO_NIVEL_2_LP,PC_ALUNO_NIVEL_3_LP,PC_ALUNO_NIVEL_4_LP,PC_ALUNO_NIVEL_5_LP,PC_ALUNO_NIVEL_6_LP,PC_ALUNO_NIVEL_7_LP,PC_ALUNO_NIVEL_8_LP
0,2024,11,RO,2,5,62.62,745.52,2.29,4.01,8.76,9.02,22.13,33.80,14.99,3.70,1.29
1,2024,11,RO,2,3,62.56,745.38,2.34,4.00,8.82,9.06,22.07,33.81,14.97,3.67,1.26
2,2024,11,RO,2,2,63.47,747.36,1.65,4.18,7.99,8.55,22.87,33.68,15.33,4.10,1.66
3,2024,12,AC,2,5,51.38,739.15,4.96,6.57,4.36,13.92,26.07,24.23,14.81,4.14,0.93
4,2024,12,AC,2,2,55.50,743.17,3.81,6.08,4.10,12.66,25.35,25.95,16.10,4.71,1.23


In [54]:
# ============================================================
# EXPLORAÇÃO DA PLANILHA DE METAS MUNICIPAIS - 2024
# ============================================================
#
# Carrega a planilha XLSX de resultados e metas municipais de 2024,
# remove linhas de rodapé e exibe volume, colunas e amostra inicial.
#
# ============================================================
# LEITURA DA PLANILHA
# ============================================================

caminho_xlsx = EXTRAIDOS_DIR / "2024" / "resultados_e_metas_municipios_2024.xlsx"

df_mun_meta = pd.read_excel(
    caminho_xlsx,
    sheet_name="Divulgação Alfabet Municipio",
    header=1,
    dtype=str
)

# ============================================================
# AJUSTE DE RODAPÉ
# ============================================================

df_mun_meta = df_mun_meta[df_mun_meta["CO_UF"].notna()].copy()

# ============================================================
# INSPEÇÃO INICIAL
# ============================================================

print(f"Linhas : {len(df_mun_meta):,}")
print(f"Colunas: {df_mun_meta.shape[1]}")

print("\nColunas:", list(df_mun_meta.columns))

df_mun_meta.head()

Linhas : 5,352
Colunas: 17

Colunas: ['ANO', 'CO_UF', 'SG_UF', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'NO_TP_REDE', 'PC_ALUNO_ALFABETIZADO_2023', 'PC_ALUNO_ALFABETIZADO_2024', 'META_FINAL_2024', 'META_FINAL_2025', 'META_FINAL_2026', 'META_FINAL_2027', 'META_FINAL_2028', 'META_FINAL_2029', 'META_FINAL_2030', 'CO_NIVEL_ALFABETIZACAO', 'PC_AVALIADOS_LP']


,ANO,CO_UF,SG_UF,CO_MUNICIPIO,NO_MUNICIPIO,NO_TP_REDE,PC_ALUNO_ALFABETIZADO_2023,PC_ALUNO_ALFABETIZADO_2024,META_FINAL_2024,META_FINAL_2025,META_FINAL_2026,META_FINAL_2027,META_FINAL_2028,META_FINAL_2029,META_FINAL_2030,CO_NIVEL_ALFABETIZACAO,PC_AVALIADOS_LP
0,2024,11,RO,1100015,Alta Floresta D'Oeste,MUNICIPAL,64.6,67.79,67.08,69.51,71.84,74.06,76.16,78.14,80,3,89.86928104575163
1,2024,11,RO,1100023,Ariquemes,MUNICIPAL,62.3,65.62,65.22,68.02,70.71,73.25,75.65,77.9,80,3,88.759367194005
2,2024,11,RO,1100031,Cabixi,MUNICIPAL,69.1,75.88,70.85,72.53,74.15,75.71,77.21,78.64,80,4,92.5925925925926
3,2024,11,RO,1100049,Cacoal,MUNICIPAL,62.5,65.81,65.39,68.16,70.81,73.33,75.7,77.92,80,3,92.56678281068524
4,2024,11,RO,1100056,Cerejeiras,MUNICIPAL,58.5,66.81,62.09,65.53,68.81,71.91,74.81,77.51,80,3,96.44268774703558


In [55]:
# ============================================================
# EXPLORAÇÃO DO DICIONÁRIO DE VARIÁVEIS - 2024
# ============================================================
#
# Carrega a aba de dicionário de variáveis da planilha de metas
# municipais de 2024 para apoiar a interpretação dos campos.
#
# ============================================================
# LEITURA DO DICIONÁRIO
# ============================================================

dic = pd.read_excel(
    caminho_xlsx,
    sheet_name="Variáveis",
    header=None,
    usecols=[1, 2],
    names=["campo", "descricao"]
).dropna(how="all")

# ============================================================
# VISUALIZAÇÃO
# ============================================================

dic

,campo,descricao
1,ANO DA AVALIAÇÃO,Ano de realização da avaliação
2,CÓDIGO UF,Código da UF a que pertence o município
3,SIGLA UF,Sigla da UF a que pertence o município
4,CÓDIGO MUNICÍPIO,Código do município
5,NOME DO MUNICÍPIO,Nome do município
6,REDE,Rede de realização: todos os resultados se ref...
7,PERCENTUAL DE ALUNOS ALFABETIZADOS - 2023 (1),Percentual de alunos alfabetizados: percentual...
8,PERCENTUAL DE ALUNOS ALFABETIZADOS - 2024 (1),Percentual de alunos alfabetizados: percentual...
9,META 2024 (2),Meta estabelecida para o município em 2024
10,META 2025,Meta estabelecida para o município em 2025


In [56]:
# ============================================================
# EXPLORAÇÃO DA PLANILHA DE METAS POR UF - 2024
# ============================================================
#
# Carrega a planilha XLSX de resultados e metas por unidade
# federativa em 2024, remove linhas de rodapé e exibe uma
# amostra inicial.
#
# ============================================================
# LEITURA DA PLANILHA
# ============================================================

caminho_xlsx = EXTRAIDOS_DIR / "2024" / "resultados_e_metas_ufs_2024_2.xlsx"

df_uf_meta = pd.read_excel(
    caminho_xlsx,
    sheet_name="Divulgação Alfabet UF e Brasil",
    header=1,
    dtype=str
)

# ============================================================
# AJUSTE DE RODAPÉ
# ============================================================

df_uf_meta = df_uf_meta[df_uf_meta["CD_UF"].notna()].copy()

# ============================================================
# INSPEÇÃO INICIAL
# ============================================================

print(f"Linhas : {len(df_uf_meta):,}")
print(f"Colunas: {df_uf_meta.shape[1]}")
print("\nColunas:", list(df_uf_meta.columns))

df_uf_meta.head()

Linhas : 27
Colunas: 15

Colunas: ['ANO', 'CD_UF', 'SIGLA_UF', 'NOME_UF', 'REDE', 'PC_ALUNO_ALFABETIZADO_2023', 'PC_ALUNO_ALFABETIZADO_2024', 'META_FINAL_2024', 'META_FINAL_2025', 'META_FINAL_2026', 'META_FINAL_2027', 'META_FINAL_2028', 'META_FINAL_2029', 'META_FINAL_2030', 'PC_AVALIADOS_LP']


,ANO,CD_UF,SIGLA_UF,NOME_UF,REDE,PC_ALUNO_ALFABETIZADO_2023,PC_ALUNO_ALFABETIZADO_2024,META_FINAL_2024,META_FINAL_2025,META_FINAL_2026,META_FINAL_2027,META_FINAL_2028,META_FINAL_2029,META_FINAL_2030,PC_AVALIADOS_LP
1,2024,12,AC,Acre,PÚBLICA,-,51.38,-,56.9,62.2,67.3,72,76.2,> 80,80.86922165152114
2,2024,27,AL,Alagoas,PÚBLICA,43.88,48.63,49.7,55.5,61.1,66.5,71.5,76,> 80,93.78396840010691
3,2024,13,AM,Amazonas,PÚBLICA,52.2,49.17,56.8,61.3,65.6,69.6,73.4,76.9,> 80,79.49477411227998
4,2024,16,AP,Amapá,PÚBLICA,41.56,46.62,47.6,53.8,59.9,65.6,70.9,75.8,> 80,89.14149443561207
5,2024,29,BA,Bahia,PÚBLICA,36.8,35.96,43.4,50.2,57.1,63.6,69.8,75.2,> 80,90.04421141883181


In [57]:
# ============================================================
# EXPLORAÇÃO DO DICIONÁRIO DE VARIÁVEIS POR UF - 2024
# ============================================================
#
# Carrega a aba de dicionário de variáveis da planilha de metas
# por unidade federativa de 2024.
#
# ============================================================
# LEITURA DO DICIONÁRIO
# ============================================================

dic_uf = (
    pd.read_excel(
        caminho_xlsx,
        sheet_name="variáveis",
        header=None,
        usecols="C:D"
    )
    .dropna(how="all")
    .rename(columns={2: "variavel", 3: "descricao"})
    .reset_index(drop=True)
)

# ============================================================
# VISUALIZAÇÃO
# ============================================================

dic_uf

,variavel,descricao
0,ANO DA AVALIAÇÃO,Ano de realização da avaliação
1,CÓDIGO UF,Código da UF
2,SIGLA UF,Sigla da UF
3,NOME UF,Nome da UF
4,REDE,Rede de realização: todos os resultados se ref...
5,PERCENTUAL DE ALUNOS ALFABETIZADOS Saeb 2019,Percentual de alunos alfabetizados Saeb 2019: ...
6,PERCENTUAL DE ALUNOS ALFABETIZADOS Saeb 2021,Percentual de alunos alfabetizados Saeb 2021: ...
7,PERCENTUAL DE ALUNOS ALFABETIZADOS Sistemas es...,Percentual de alunos alfabetizados Sistemas es...
8,PERCENTUAL DE ALUNOS ALFABETIZADOS Sistemas es...,Percentual de alunos alfabetizados Sistemas es...
9,META 2024,Meta estabelecida para o estado ou Brasil em 2024


## 2025

Nesta seção, repetimos a exploração inicial para os arquivos referentes ao ano de 2025, usando a mesma lógica aplicada nos anos anteriores.

In [58]:
# ============================================================
# EXPLORAÇÃO DO ARQUIVO DE MUNICÍPIOS - 2025
# ============================================================
#
# Carrega o arquivo TS_MUNICIPIO.csv de 2025 e exibe volume,
# estrutura, tipos de dados e uma amostra inicial.
#
# ============================================================
# LEITURA DO ARQUIVO
# ============================================================

caminho_municipio = EXTRAIDOS_DIR / "2025" / "TS_MUNICIPIO.csv"
df_mun = pd.read_csv(caminho_municipio, **PARAMS_LEITURA)

# ============================================================
# INSPEÇÃO INICIAL
# ============================================================

print(f"Linhas : {len(df_mun):,}")
print(f"Colunas: {df_mun.shape[1]}")

print("\n--- info(): schema, não nulos e memória ---")
df_mun.info()

print("\n--- Primeiras 5 linhas ---")
df_mun.head()

Linhas : 12,416
Colunas: 18

--- info(): schema, não nulos e memória ---
<class 'pandas.DataFrame'>
RangeIndex: 12416 entries, 0 to 12415
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   NU_ANO_AVALIACAO       12416 non-null  int64  
 1   CO_UF                  12416 non-null  int64  
 2   SG_UF                  12416 non-null  str    
 3   CO_MUNICIPIO           12416 non-null  int64  
 4   NO_MUNICIPIO           12416 non-null  str    
 5   TP_SERIE               12416 non-null  int64  
 6   ID_TIPO_REDE           12416 non-null  int64  
 7   PC_ALUNO_ALFABETIZADO  12416 non-null  float64
 8   VL_MEDIA_LP            12416 non-null  float64
 9   PC_ALUNO_NIVEL_0_LP    12416 non-null  float64
 10  PC_ALUNO_NIVEL_1_LP    12416 non-null  float64
 11  PC_ALUNO_NIVEL_2_LP    12416 non-null  float64
 12  PC_ALUNO_NIVEL_3_LP    12416 non-null  float64
 13  PC_ALUNO_NIVEL_4_LP    12416 non-null  float

,NU_ANO_AVALIACAO,CO_UF,SG_UF,CO_MUNICIPIO,NO_MUNICIPIO,TP_SERIE,ID_TIPO_REDE,PC_ALUNO_ALFABETIZADO,VL_MEDIA_LP,PC_ALUNO_NIVEL_0_LP,PC_ALUNO_NIVEL_1_LP,PC_ALUNO_NIVEL_2_LP,PC_ALUNO_NIVEL_3_LP,PC_ALUNO_NIVEL_4_LP,PC_ALUNO_NIVEL_5_LP,PC_ALUNO_NIVEL_6_LP,PC_ALUNO_NIVEL_7_LP,PC_ALUNO_NIVEL_8_LP
0,2025,11,RO,1100015,Alta Floresta D'Oeste,2,3,78.15,763.2938,0.71,2.92,5.59,7.04,10.22,32.52,25.25,11.54,4.21
1,2025,11,RO,1100015,Alta Floresta D'Oeste,2,5,78.15,763.2938,0.71,2.92,5.59,7.04,10.22,32.52,25.25,11.54,4.21
2,2025,11,RO,1100023,Ariquemes,2,5,79.50,765.1730,1.61,2.40,3.98,6.47,10.20,30.22,28.52,13.14,3.45
3,2025,11,RO,1100023,Ariquemes,2,3,79.50,765.1730,1.61,2.40,3.98,6.47,10.20,30.22,28.52,13.14,3.45
4,2025,11,RO,1100031,Cabixi,2,5,86.89,768.9794,0.00,0.00,4.92,3.28,8.20,42.63,32.79,4.91,3.28


In [ ]:
# ============================================================
# EXPLORAÇÃO DO ARQUIVO DE ALUNOS - 2025
# ============================================================
#
# Carrega uma amostra do arquivo TS_ALUNO.csv de 2025, selecionando
# apenas as colunas relevantes para a análise.
#
# Em 2025, o arquivo passou a conter novas colunas em relação aos
# anos anteriores, provavelmente relacionadas às respostas dos alunos
# no teste. Como essas colunas não fazem parte do escopo atual da
# pipeline, usamos usecols para ler apenas os campos necessários.
#
# Essa seleção reduz o consumo de memória e mantém a exploração
# alinhada ao conjunto de variáveis utilizado nas próximas etapas.
#
# ============================================================
# LEITURA DO ARQUIVO
# ============================================================

caminho_aluno = EXTRAIDOS_DIR / "2025" / "TS_ALUNO.csv"

# Colunas relevantes para a exploração e para as próximas etapas da pipeline.
COLS = [
    "NU_ANO_AVALIACAO",
    "CO_UF",
    "SG_UF",
    "ID_ALUNO",
    "TP_SERIE",
    "ID_ESCOLA",
    "TP_DEPENDENCIA",
    "CO_MUNICIPIO",
    "NO_MUNICIPIO",
    "IN_PRESENCA_LP",
    "IN_PREENCHIMENTO_LP",
    "CO_CADERNO_LP",
    "VL_PESO_ALUNO_LP",
    "VL_PROFICIENCIA_LP",
    "IN_ALFABETIZADO",
]

# Lê uma amostra do arquivo, mantendo apenas as colunas selecionadas.
df_aluno = pd.read_csv(
    caminho_aluno,
    usecols=COLS,
    nrows=5000,
    **PARAMS_LEITURA
)

# ============================================================
# INSPEÇÃO INICIAL
# ============================================================

print(f"Linhas lidas : {len(df_aluno):,}")
print(f"Colunas     : {df_aluno.shape[1]}")

# A diferença de não nulos entre colunas ajuda a identificar campos
# vazios associados a alunos ausentes.
print("\n--- info(): repare a diferença de não nulos entre colunas ---")
df_aluno.info()

print("\n--- Primeiras 5 linhas ---")
display(df_aluno.head())

Linhas lidas : 5,000
Colunas     : 15

--- info(): repare a diferença de não nulos entre colunas ---
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   NU_ANO_AVALIACAO     5000 non-null   int64  
 1   CO_UF                5000 non-null   int64  
 2   SG_UF                5000 non-null   str    
 3   ID_ALUNO             5000 non-null   int64  
 4   TP_SERIE             5000 non-null   int64  
 5   ID_ESCOLA            5000 non-null   int64  
 6   TP_DEPENDENCIA       5000 non-null   int64  
 7   CO_MUNICIPIO         5000 non-null   int64  
 8   NO_MUNICIPIO         5000 non-null   str    
 9   IN_PRESENCA_LP       5000 non-null   int64  
 10  IN_PREENCHIMENTO_LP  5000 non-null   int64  
 11  CO_CADERNO_LP        5000 non-null   int64  
 12  VL_PESO_ALUNO_LP     4571 non-null   float64
 13  VL_PROFICIENCIA_LP   4571 non-null   float64
 14

,NU_ANO_AVALIACAO,CO_UF,SG_UF,ID_ALUNO,TP_SERIE,ID_ESCOLA,TP_DEPENDENCIA,CO_MUNICIPIO,NO_MUNICIPIO,IN_PRESENCA_LP,IN_PREENCHIMENTO_LP,CO_CADERNO_LP,VL_PESO_ALUNO_LP,VL_PROFICIENCIA_LP,IN_ALFABETIZADO
0,2025,11,RO,11000677,2,60000060,3,1100023,Ariquemes,1,1,8,1.5,781.776332,1
1,2025,11,RO,11000678,2,60000060,3,1100023,Ariquemes,1,1,8,1.5,717.877346,0
2,2025,11,RO,11000679,2,60000060,3,1100023,Ariquemes,1,1,9,1.5,657.836805,0
3,2025,11,RO,11000680,2,60000060,3,1100023,Ariquemes,1,1,14,1.5,765.290294,1
4,2025,11,RO,11000681,2,60000060,3,1100023,Ariquemes,1,1,10,1.5,835.506362,1


,NU_ANO_AVALIACAO,CO_UF,SG_UF,ID_ALUNO,TP_SERIE,ID_ESCOLA,TP_DEPENDENCIA,CO_MUNICIPIO,NO_MUNICIPIO,IN_PRESENCA_LP,IN_PREENCHIMENTO_LP,CO_CADERNO_LP,VL_PESO_ALUNO_LP,VL_PROFICIENCIA_LP,IN_ALFABETIZADO
0,2025,11,RO,11000677,2,60000060,3,1100023,Ariquemes,1,1,8,1.5,781.776332,1
1,2025,11,RO,11000678,2,60000060,3,1100023,Ariquemes,1,1,8,1.5,717.877346,0
2,2025,11,RO,11000679,2,60000060,3,1100023,Ariquemes,1,1,9,1.5,657.836805,0
3,2025,11,RO,11000680,2,60000060,3,1100023,Ariquemes,1,1,14,1.5,765.290294,1
4,2025,11,RO,11000681,2,60000060,3,1100023,Ariquemes,1,1,10,1.5,835.506362,1


In [29]:
# ============================================================
# VERIFICAÇÃO DE PROFICIÊNCIA NULA EM ALUNOS PRESENTES - 2025
# ============================================================
#
# Esta célula verifica se existem alunos marcados como presentes na
# prova de Língua Portuguesa, mas sem valor de proficiência preenchido.
#
# Essa checagem ajuda a entender a relação entre presença,
# preenchimento da prova e cálculo da proficiência.
#
# Observação:
#
#   Como df_aluno foi carregado com nrows=5000 na célula anterior,
#   o resultado abaixo representa apenas a amostra lida, não o arquivo
#   completo.
#
# ============================================================
# FILTRO DE INCONSISTÊNCIA APARENTE
# ============================================================

presentes_sem_prof = df_aluno[
    (df_aluno["IN_PRESENCA_LP"] == 1)
    & (df_aluno["VL_PROFICIENCIA_LP"].isna())
]

print("Presentes com proficiência nula:", len(presentes_sem_prof))

# ============================================================
# AGRUPAMENTO PARA INVESTIGAÇÃO
# ============================================================

# Agrupa os casos encontrados por ano e indicador de preenchimento.
# Isso ajuda a verificar se a ausência de proficiência está associada
# ao não preenchimento da prova.
(
    presentes_sem_prof
    .groupby(["NU_ANO_AVALIACAO", "IN_PREENCHIMENTO_LP"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["NU_ANO_AVALIACAO", "IN_PREENCHIMENTO_LP"])
)

Presentes com proficiência nula: 3316


,NU_ANO_AVALIACAO,IN_PREENCHIMENTO_LP,count
0,2025,0,3316


In [33]:
# ============================================================
# AMOSTRA DOS ALUNOS PRESENTES SEM PROFICIÊNCIA - 2025
# ============================================================
#
# Esta célula exibe alguns registros de alunos presentes na prova,
# mas sem proficiência calculada.
#
# O objetivo é inspecionar, lado a lado, campos relacionados à
# presença, preenchimento, peso do aluno, proficiência e indicador
# de alfabetização.
#
# ============================================================
# VISUALIZAÇÃO DOS CAMPOS RELEVANTES
# ============================================================

presentes_sem_prof[
        [
            "NU_ANO_AVALIACAO",
            "IN_PRESENCA_LP",
            "IN_PREENCHIMENTO_LP",
            "VL_PESO_ALUNO_LP",
            "VL_PROFICIENCIA_LP",
            "IN_ALFABETIZADO",
        ]
    ].head(10)

,NU_ANO_AVALIACAO,IN_PRESENCA_LP,IN_PREENCHIMENTO_LP,VL_PESO_ALUNO_LP,VL_PROFICIENCIA_LP,IN_ALFABETIZADO
23331,2025,1,0,NaN,NaN,0
23695,2025,1,0,NaN,NaN,0
23696,2025,1,0,NaN,NaN,0
24823,2025,1,0,NaN,NaN,0
24824,2025,1,0,NaN,NaN,0
24825,2025,1,0,NaN,NaN,0
24826,2025,1,0,NaN,NaN,0
24827,2025,1,0,NaN,NaN,0
24828,2025,1,0,NaN,NaN,0
24829,2025,1,0,NaN,NaN,0


In [38]:
# ============================================================
# EXPLORAÇÃO DO ARQUIVO DE ESTADOS - 2025
# ============================================================
#
# Carrega o arquivo TS_ESTADO.csv de 2025 e exibe volume,
# resumo estatístico das medidas reais e uma amostra inicial.
#
# ============================================================
# LEITURA DO ARQUIVO
# ============================================================

caminho_estado = EXTRAIDOS_DIR / "2025" / "TS_ESTADO.csv"
df_est = pd.read_csv(caminho_estado, **PARAMS_LEITURA)

# ============================================================
# INSPEÇÃO INICIAL
# ============================================================

print(f"Linhas : {len(df_est):,}")
print(f"Colunas: {df_est.shape[1]}")

# ============================================================
# RESUMO ESTATÍSTICO
# ============================================================

# Seleciona apenas medidas analíticas reais.
# Códigos numéricos, como CO_UF, TP_SERIE e ID_TIPO_REDE,
# não possuem média ou desvio com significado analítico.
COLS_MEDIDA = ["PC_ALUNO_ALFABETIZADO", "VL_MEDIA_LP"]

print("\n--- Resumo estatístico: apenas medidas ---")
display(df_est[COLS_MEDIDA].describe())

print("\n--- Primeiras 5 linhas ---")
display(df_est.head())

Linhas : 80
Colunas: 16

--- Resumo estatístico: apenas medidas ---


,PC_ALUNO_ALFABETIZADO,VL_MEDIA_LP
count,80.000000,80.000000
mean,65.306750,753.072216
std,10.998867,15.570643
min,39.240000,723.496400
25%,57.477500,743.144650
50%,64.550000,750.225700
75%,75.195000,763.543900
max,84.090000,792.815400



--- Primeiras 5 linhas ---


,NU_ANO_AVALIACAO,CO_UF,SG_UF,TP_SERIE,ID_TIPO_REDE,PC_ALUNO_ALFABETIZADO,VL_MEDIA_LP,PC_ALUNO_NIVEL_0_LP,PC_ALUNO_NIVEL_1_LP,PC_ALUNO_NIVEL_2_LP,PC_ALUNO_NIVEL_3_LP,PC_ALUNO_NIVEL_4_LP,PC_ALUNO_NIVEL_5_LP,PC_ALUNO_NIVEL_6_LP,PC_ALUNO_NIVEL_7_LP,PC_ALUNO_NIVEL_8_LP
0,2025,11,RO,2,2,77.66,763.9429,1.04,1.63,4.97,7.54,13.13,30.33,26.43,11.12,3.81
1,2025,11,RO,2,5,75.30,761.7828,1.73,2.49,5.07,7.93,12.37,29.96,25.65,11.10,3.70
2,2025,11,RO,2,3,75.16,761.6562,1.77,2.54,5.08,7.96,12.32,29.94,25.60,11.10,3.69
3,2025,12,AC,2,5,67.99,752.7504,3.08,4.18,8.73,4.37,19.49,27.27,20.94,9.39,2.56
4,2025,12,AC,2,2,71.12,756.9630,2.54,3.77,7.75,4.19,17.84,26.70,23.21,10.55,3.45


In [59]:
# ============================================================
# EXPLORAÇÃO DA PLANILHA DE METAS MUNICIPAIS - 2025
# ============================================================
#
# Carrega a planilha XLSX de resultados e metas municipais de 2025,
# remove linhas de rodapé e exibe volume, colunas e amostra inicial.
#
# ============================================================
# LEITURA DA PLANILHA
# ============================================================

caminho_xlsx = EXTRAIDOS_DIR / "2025" / "resultados_e_metas_municipios_2025_v2.xlsx"

df_mun_meta = pd.read_excel(
    caminho_xlsx,
    sheet_name="Divulgação Alfabet Municipio",
    header=1,
    dtype=str
)

# ============================================================
# AJUSTE DE RODAPÉ
# ============================================================

df_mun_meta = df_mun_meta[df_mun_meta["CO_UF"].notna()].copy()

# ============================================================
# INSPEÇÃO INICIAL
# ============================================================

print(f"Linhas : {len(df_mun_meta):,}")
print(f"Colunas: {df_mun_meta.shape[1]}")

print("\nColunas:", list(df_mun_meta.columns))
df_mun_meta.head()

Linhas : 5,466
Colunas: 18

Colunas: ['ANO', 'CO_UF', 'SG_UF', 'CO_MUNICIPIO', 'NO_MUNICIPIO', 'NO_TP_REDE', 'PC_ALUNO_ALFABETIZADO_2023', 'PC_ALUNO_ALFABETIZADO_2024', 'PC_ALUNO_ALFABETIZADO_2025', 'META_FINAL_2024', 'META_FINAL_2025', 'META_FINAL_2026', 'META_FINAL_2027', 'META_FINAL_2028', 'META_FINAL_2029', 'META_FINAL_2030', 'CO_NIVEL_ALFABETIZACAO', 'PC_AVALIADOS_LP']


,ANO,CO_UF,SG_UF,CO_MUNICIPIO,NO_MUNICIPIO,NO_TP_REDE,PC_ALUNO_ALFABETIZADO_2023,PC_ALUNO_ALFABETIZADO_2024,PC_ALUNO_ALFABETIZADO_2025,META_FINAL_2024,META_FINAL_2025,META_FINAL_2026,META_FINAL_2027,META_FINAL_2028,META_FINAL_2029,META_FINAL_2030,CO_NIVEL_ALFABETIZACAO,PC_AVALIADOS_LP
0,2025,11,RO,1100015,Alta Floresta D'Oeste,MUNICIPAL,65,68,78,67,70,72,74,76,78,80,4,91.53
1,2025,11,RO,1100023,Ariquemes,MUNICIPAL,62,66,80,65,68,71,73,76,78,80,4,70.07
2,2025,11,RO,1100031,Cabixi,MUNICIPAL,69,76,87,71,73,74,76,77,79,80,5,95.31
3,2025,11,RO,1100049,Cacoal,MUNICIPAL,63,66,85,65,68,71,73,76,78,80,5,91.84
4,2025,11,RO,1100056,Cerejeiras,MUNICIPAL,59,67,90,62,66,69,72,75,78,80,5,95.77


In [60]:
# ============================================================
# EXPLORAÇÃO DO DICIONÁRIO DE VARIÁVEIS - 2025
# ============================================================
#
# Carrega a aba de dicionário de variáveis da planilha de metas
# municipais de 2025 para apoiar a interpretação dos campos.
#
# ============================================================
# LEITURA DO DICIONÁRIO
# ============================================================

dic = pd.read_excel(
    caminho_xlsx,
    sheet_name="Variáveis",
    header=None,
    usecols=[1, 2],
    names=["campo", "descricao"]
).dropna(how="all")

# ============================================================
# VISUALIZAÇÃO
# ============================================================

dic

,campo,descricao
1,ANO DA AVALIAÇÃO,Ano de realização da avaliação
2,CÓDIGO UF,Código da UF a que pertence o município
3,SIGLA UF,Sigla da UF a que pertence o município
4,CÓDIGO MUNICÍPIO,Código do município
5,NOME DO MUNICÍPIO,Nome do município
6,REDE,Rede de realização: todos os resultados se ref...
7,PERCENTUAL DE ALUNOS ALFABETIZADOS - 2023 (1),Percentual de alunos alfabetizados: percentual...
8,PERCENTUAL DE ALUNOS ALFABETIZADOS - 2024 (1),Percentual de alunos alfabetizados: percentual...
9,PERCENTUAL DE ALUNOS ALFABETIZADOS - 2025 (1),Percentual de alunos alfabetizados: percentual...
10,META 2024 (2),Meta estabelecida para o município em 2024


In [61]:
# ============================================================
# EXPLORAÇÃO DA PLANILHA DE METAS POR UF - 2025
# ============================================================
#
# Carrega a planilha XLSX de resultados e metas por unidade
# federativa em 2025, remove linhas de rodapé e exibe uma
# amostra inicial.
#
# ============================================================
# LEITURA DA PLANILHA
# ============================================================

caminho_xlsx = EXTRAIDOS_DIR / "2025" / "resultados_e_metas_ufs_2025_v1.xlsx"

df_uf_meta = pd.read_excel(
    caminho_xlsx,
    sheet_name="Divulgação Alfabet UF e Brasil",
    header=1,
    dtype=str
)

# ============================================================
# AJUSTE DE RODAPÉ
# ============================================================

df_uf_meta = df_uf_meta[df_uf_meta["CD_UF"].notna()].copy()

# ============================================================
# INSPEÇÃO INICIAL
# ============================================================

print(f"Linhas : {len(df_uf_meta):,}")
print(f"Colunas: {df_uf_meta.shape[1]}")
print("\nColunas:", list(df_uf_meta.columns))

df_uf_meta.head()

Linhas : 27
Colunas: 16

Colunas: ['ANO', 'CD_UF', 'SIGLA_UF', 'NOME_UF', 'REDE', 'PC_ALUNO_ALFABETIZADO_2023', 'PC_ALUNO_ALFABETIZADO_2024', 'PC_ALUNO_ALFABETIZADO_2025', 'META_FINAL_2024', 'META_FINAL_2025', 'META_FINAL_2026', 'META_FINAL_2027', 'META_FINAL_2028', 'META_FINAL_2029', 'META_FINAL_2030', 'PC_AVALIADOS_LP']


,ANO,CD_UF,SIGLA_UF,NOME_UF,REDE,PC_ALUNO_ALFABETIZADO_2023,PC_ALUNO_ALFABETIZADO_2024,PC_ALUNO_ALFABETIZADO_2025,META_FINAL_2024,META_FINAL_2025,META_FINAL_2026,META_FINAL_2027,META_FINAL_2028,META_FINAL_2029,META_FINAL_2030,PC_AVALIADOS_LP
1,2025,12,AC,Acre,PÚBLICA,NaN,51,68,NaN,57,62,67,72,76,> 80,82
2,2025,27,AL,Alagoas,PÚBLICA,44,49,64,50,56,61,67,72,76,> 80,94
3,2025,13,AM,Amazonas,PÚBLICA,52,49,57,57,61,66,70,73,77,> 80,86
4,2025,16,AP,Amapá,PÚBLICA,42,47,60,48,54,60,66,71,76,> 80,88
5,2025,29,BA,Bahia,PÚBLICA,37,36,55,43,50,57,64,70,75,> 80,87


In [62]:
# ============================================================
# EXPLORAÇÃO DO DICIONÁRIO DE VARIÁVEIS POR UF - 2025
# ============================================================
#
# Carrega a aba de dicionário de variáveis da planilha de metas
# por unidade federativa de 2025.
#
# ============================================================
# LEITURA DO DICIONÁRIO
# ============================================================

dic_uf = (
    pd.read_excel(
        caminho_xlsx,
        sheet_name="variáveis",
        header=None,
        usecols="C:D"
    )
    .dropna(how="all")
    .rename(columns={2: "variavel", 3: "descricao"})
    .reset_index(drop=True)
)

# ============================================================
# VISUALIZAÇÃO
# ============================================================

dic_uf

,variavel,descricao
0,ANO DA AVALIAÇÃO,Ano de realização da avaliação
1,CÓDIGO UF,Código da UF
2,SIGLA UF,Sigla da UF
3,NOME UF,Nome da UF
4,REDE,Rede de realização: todos os resultados se ref...
5,PERCENTUAL DE ALUNOS ALFABETIZADOS Saeb 2019,Percentual de alunos alfabetizados Saeb 2019: ...
6,PERCENTUAL DE ALUNOS ALFABETIZADOS Saeb 2021,Percentual de alunos alfabetizados Saeb 2021: ...
7,PERCENTUAL DE ALUNOS ALFABETIZADOS Sistemas es...,Percentual de alunos alfabetizados Sistemas es...
8,PERCENTUAL DE ALUNOS ALFABETIZADOS Sistemas es...,Percentual de alunos alfabetizados Sistemas es...
9,META 2024,Meta estabelecida para o estado ou Brasil em 2024
